In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#1. leemos el archivo JSON multilinea
# Define el la estructura personName
movie_cast_schema = StructType(fields = [
    StructField("movieId", IntegerType(), True),
    StructField("personId", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genderId", IntegerType(), True),
    StructField("castOrder", IntegerType(), True)
])

# Cargamos el archivo utilizando la estructura definida
movie_cast_df = spark.read\
    .schema(movie_cast_schema)\
    .option("multiLine", "true")\
    .json(f"{bronze_folder_path}/movie_cast.json")

# Mostramos el resultado
display(movie_cast_df)


In [0]:
#Paso 2 - Renombrar, añadir y dar formato a las columnas requeridas
movie_cast_renamed_df = movie_cast_df\
    .withColumnRenamed("movieId", "movie_id")\
    .withColumnRenamed("personId", "person_id")\
    .withColumnRenamed("characterName", "character_name")

movie_cast_renamed_df = add_ingestion_date(movie_cast_renamed_df)
movie_cast_renamed_df = add_env(movie_cast_renamed_df)

display(movie_cast_renamed_df)

In [0]:
#Paso 3 - Seleccionar las columnas que se requieren 
movie_cast_final_df = movie_cast_renamed_df.select(col("movie_id"), col("person_id"), col("character_name"), col("ingestion_date"), col("env"))
display(movie_cast_final_df)


In [0]:
#Paso 4 - Guardar datos en datalake en formato parket 

movie_cast_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/movie_cast")

df = spark.read.parquet(f"{silver_folder_path}/movie_cast")
display(df)


In [0]:
dbutils.notebook.exit("El notebook 07. Ingestion File movie_cast, termino correctamente")